# Extra: selection of TF-IDF components based on entropy

This notebook is a small extension of the original project. Here we want to further analyze the network built using tf-idf embeddings of abstracts. In particular, since tf-idf embedding vectors have a huge number of components, we want to see if filtering some of these components can lead to a reduction of the noise present in the embedding vectors, and thus bring to an improvement of classification performances.

The selection of components is based on entropy: for each component of the vector embeddings, the information entropy is calculated across all papers is calculated using:

\begin{equation}
    H = -\sum_{i=1}^N p_ilog(p_i)
\end{equation}

where $N = 25877$ is the total number of papers, while $p_i$ represents the value of a certain component of embedding vectors in paper $i$ when the values of that component are normalized across all papers so that $\sum_{i=1}^N p_i = 1$.

After calculating the value of entropy for all embedding components, only the components with entropy higher than a certain threshold are kept. Finally, the distance matrix, the adjacency matrix and the corresponding network are built from tf-idf embeddings filtered in this way, and all the algorithms already employed in the original project are tried to divide the network into communities, to see if we have an improvement of classification after the selection of components.

## Entropy calculation

First of all we want to compute the entropy of all components of embedding vectors, and save in a npz file the array containing all values of entropy.

In [8]:
from components_selection import entropy_component
from scipy.sparse import load_npz
import numpy as np
import pandas as pd
import networkx as nx

In [6]:
tf_idf_embeddings = load_npz("../embeddings/abstract_embeddings_tfidf.npz")

In [ ]:
for i in range(223):
    #select 250 columns, since 223 * 250 = 55750, number of columns of the original matrix
    batch_of_columns = tf_idf_embeddings[:, (i * 250):((i+1) * 250)].toarray()

    entropy = entropy_component(batch_of_columns)

    #append the arrays obtained in the different cycles
    if i == 0:
        full_entropy_array = entropy
    else:
        full_entropy_array = np.concatenate((full_entropy_array, entropy))

#save the array in the npz format
np.savez("./entropy_array.npz", full_entropy_array)

/home/riccardo/uni_projects/complex_networks/Abstract_network/extra/components_selection.py:32: RuntimeWarning: divide by zero encountered in log
  embedding_matrix_normalized = np.where(np.isclose(embedding_matrix_normalized, 0.), 0., (embedding_matrix_normalized * np.log(embedding_matrix_normalized)) * (-1))
/home/riccardo/uni_projects/complex_networks/Abstract_network/extra/components_selection.py:32: RuntimeWarning: invalid value encountered in multiply
  embedding_matrix_normalized = np.where(np.isclose(embedding_matrix_normalized, 0.), 0., (embedding_matrix_normalized * np.log(embedding_matrix_normalized)) * (-1))


Now load the array with entropies, and filter the components of the tf-idf embeddings keeping only those with entropy greater than 1 (this threshold on entropy is selected arbitrarily, later we will see how different values of threshold influence the structure of the resulting network).

In [7]:
entropy_array = np.load("./entropy_array.npz")['arr_0']
filtered_embeddings = tf_idf_embeddings[:, np.where(entropy_array > 1)[0]]

In [5]:
print(np.max(entropy_array))
print(np.min(entropy_array))

10.028510818552421
0.0


As a first analysis, we can print the maximum and minimum values of entropy of the embedding components: as we expect, the minimum value of entropy is 0: this happens when the component is different from 0 only in one paper (so it is a word that appears only in a certain paper).

## Build network and split into communities

Now we can build the distance matrix, the adjacency matrix, the relative network using the same functions we used for the network with all the components.

First, we try different values of the threshold on the distance matrix, as we did with the network built using all components.

In [3]:
from components_selection import sweep_connected_components

sweep_connected_components(embedding_matrix = filtered_embeddings)

Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration


Results are similar to those obtained using all components, but in general we get a lower number of components. So we use again a threshold of 0.2.

Next, using always the threshold distance of 0.2, we try different values of threshold entropy to filter the components of the embeddings. In particular, we use ten different thresholds in the range 1-8 (the range is selected considering the minimum and maximum value of entropy).

In [1]:
from components_selection import sweep_entropy_threshold

sweep_entropy_threshold()

Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration
Iteration


In [3]:
pd.read_csv("./results/connected_components_entropy.csv")

,Unnamed: 0,Threshold,Embedding_components,Connected_components,Largest_component
0,0,1.000000,22989.0,298.0,25572.0
1,1,1.444444,17691.0,199.0,25675.0
2,2,1.888889,14455.0,148.0,25728.0
3,3,2.333333,11615.0,119.0,25759.0
4,4,2.777778,9382.0,76.0,25801.0
5,5,3.222222,7596.0,52.0,25826.0
6,6,3.666667,6048.0,25.0,25853.0
7,7,4.111111,4755.0,4.0,25874.0
8,8,4.555556,3679.0,4.0,25874.0
9,9,5.000000,2737.0,1.0,25877.0


The higher the threshold on entropy, the lower number of connected components we have: this means that different vectors are more similar one to the other.

We choose then a threshold on entropy of 4.

In [9]:
from components_selection import build_adjacency_matrix

#filter embeddings and build the relative adjacency matrix
filtered_embeddings = tf_idf_embeddings[:, np.where(entropy_array > 4)[0]]
print("Number of remaining components: " + str(filtered_embeddings.shape[1]))
adjacency_matrix = build_adjacency_matrix(filtered_embeddings, threshold = 0.2)

#build the network corresponding to the adjacency matrix, and count the number of links, the number of connected components
#and the largest connected component
G = nx.from_scipy_sparse_array(adjacency_matrix)
print("Number of links: " + str(G.size()))
print("Number of connected components: " + str(nx.number_connected_components(G)))
print("Size of the largest connected component: " + str(max([len(c) for c in list(nx.connected_components(G))])))

Number of remaining components: 5064
Number of links: 2894107
Number of connected components: 12
Size of the largest connected component: 25866
